# Section 1: Install and Import Libraries


In [16]:
!pip install yfinance plotly scipy -q

import yfinance as yf
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Fetch current risk-free rate (3-month T-bill) while yf is available
tbill = yf.download('^IRX', period='5d', auto_adjust=True, progress = False)['Close']
RISK_FREE_RATE = float(tbill.dropna().iloc[-1]) / 100

print("All libraries loaded successfully!")
print(f"Risk-free rate fetched: {RISK_FREE_RATE:.2%}")

All libraries loaded successfully!
Risk-free rate fetched: 3.61%


# Section 2: CONFIGURATION — EDIT THIS TO CUSTOMIZE

In [17]:
# Supported: Individual stocks (AAPL), ETFs (SPY, VOO), Bond funds (TLT, AGG), REITs (VNQ)
# Not recommended: Commodity futures (GC=F) or Crypto (BTC-USD) due to data issues
TICKERS = ['SPY', 'TLT', 'GOOGL', 'JPM', 'VNQ']

# Date range for historical data — dynamically set to today's date
from datetime import date
from dateutil.relativedelta import relativedelta

END_DATE   = date.today().strftime('%Y-%m-%d')
START_DATE = (date.today() - relativedelta(years=7)).strftime('%Y-%m-%d')

# Risk-free rate — fetched dynamically in Section 1 from Yahoo Finance (^IRX, 3-month T-bill)
# Industry standard is 0.95; banks often use 0.99 under Basel regulations.
VAR_CONFIDENCE = 0.95

# Number of portfolios to simulate for the efficient frontier
NUM_PORTFOLIOS = 10000

print(f"Configuration set for: {TICKERS}")
print(f"Date range: {START_DATE} to {END_DATE}")

Configuration set for: ['SPY', 'TLT', 'GOOGL', 'JPM', 'VNQ']
Date range: 2019-03-31 to 2026-03-31


# Section 3: Fetch Historical Price Data

In [18]:
print(f"\n Fetching price data from Yahoo Finance...")

# Download adjusted closing prices for all tickers
# Adjusted close accounts for stock splits and dividends,
# making it the most accurate measure of true returns
raw_data = yf.download(TICKERS, start=START_DATE, end=END_DATE, auto_adjust=True, progress = False)
prices = raw_data['Close']

# Drop any dates where all stocks have missing data
prices.dropna(how='all', inplace=True)

print(f" Data fetched: {len(prices)} trading days from {prices.index[0].date()} to {prices.index[-1].date()}")
print(f"\n Latest prices:")
print(prices.tail(1).T.rename(columns={prices.index[-1]: 'Price ($)'}))


 Fetching price data from Yahoo Finance...
 Data fetched: 1759 trading days from 2019-04-01 to 2026-03-30

 Latest prices:
Date     Price ($)
Ticker            
GOOGL   273.500000
JPM     283.769989
SPY     631.969971
TLT      86.779999
VNQ      87.330002


# Section 4: CALCULATE DAILY RETURNS

In [19]:
# Log return = ln(today's price / yesterday's price)
# Using log returns is standard in finance because they are:
# - Time-additive (you can sum them across periods)
# - Continuously compounded (more mathematically precise)
# - Approximately normally distributed (better for statistical models)
# Note: log returns ≈ simple returns for small daily moves, but differ over longer periods
daily_returns = np.log(prices / prices.shift(1)).dropna()

print(f"\n Daily Returns Summary (annualized):")
print(f"   Mean Annual Return per stock (assuming 252 trading days/year):")
annual_returns = daily_returns.mean() * 252
for ticker, ret in annual_returns.items():
    print(f"   {ticker}: {ret:.1%}")


 Daily Returns Summary (annualized):
   Mean Annual Return per stock (assuming 252 trading days/year):
   GOOGL: 21.9%
   JPM: 17.1%
   SPY: 12.9%
   TLT: -2.4%
   VNQ: 3.9%


# Section 5: RISK METRIC — SHARPE RATIO

In [20]:
# The Sharpe Ratio measures RISK-ADJUSTED return.
# Formula: (Portfolio Return - Risk-Free Rate) / Portfolio Volatility
#
# A Sharpe Ratio of:
#   > 1.0 = Good (you're being well compensated for risk)
#   > 2.0 = Very good
#   < 0   = You'd be better off in Treasury Bills
#
# It answers: "How much extra return am I getting per unit of risk?"

annual_volatility = daily_returns.std() * np.sqrt(252)  # annualize by √252

sharpe_ratios = (annual_returns - RISK_FREE_RATE) / annual_volatility

print(f"\n Sharpe Ratios (Risk-Free Rate = {RISK_FREE_RATE:.2%}):")
for ticker in TICKERS:
    print(f"   {ticker}: {sharpe_ratios[ticker]:.2f}")


 Sharpe Ratios (Risk-Free Rate = 3.61%):
   SPY: 0.47
   TLT: -0.36
   GOOGL: 0.58
   JPM: 0.45
   VNQ: 0.01


# Section 6: RISK METRICS — VALUE AT RISK (VaR) and Conditional Value at Risk (CVaR)


In [21]:
print(f"\n Value at Risk & CVaR ({VAR_CONFIDENCE:.0%} confidence, daily):")
var_values = {}
cvar_values = {}
for ticker in TICKERS:
    # VaR — worst expected daily loss at 95% confidence
    var = np.percentile(daily_returns[ticker], (1 - VAR_CONFIDENCE) * 100)
    var_values[ticker] = var

    # CVaR — average loss on the worst 5% of days
    # This answers: "when things are bad, how bad do they get on average?"
    # CVaR is considered more accurate than VaR for tail risk
    # Basel III regulations require banks to use CVaR over VaR
    cvar = daily_returns[ticker][daily_returns[ticker] <= var].mean()
    cvar_values[ticker] = cvar

    print(f"   {ticker}: VaR {var:.2%} | CVaR {cvar:.2%}")



 Value at Risk & CVaR (95% confidence, daily):
   SPY: VaR -1.78% | CVaR -3.05%
   TLT: VaR -1.66% | CVaR -2.26%
   GOOGL: VaR -3.06% | CVaR -4.61%
   JPM: VaR -2.80% | CVaR -4.42%
   VNQ: VaR -2.09% | CVaR -3.48%


# Section 7: RISK METRIC — MAXIMUM DRAWDOWN

In [22]:
# Maximum Drawdown measures the largest peak-to-trough decline
# in portfolio value over the entire time period.
#
# Example: If a stock went from $100 → $150 → $90,
# the max drawdown is ($90 - $150) / $150 = -40%
#
# This matters because it shows the worst-case scenario an investor
# would have experienced if they bought at the peak.

print(f"\nMaximum Drawdown:")
max_drawdowns = {}
for ticker in TICKERS:
    cumulative = prices[ticker] / prices[ticker].iloc[0]
    rolling_max = cumulative.cummax()
    drawdown = (cumulative - rolling_max) / rolling_max
    max_dd = drawdown.min()
    max_drawdowns[ticker] = max_dd
    print(f"   {ticker}: {max_dd:.2%}")


Maximum Drawdown:
   SPY: -33.72%
   TLT: -48.35%
   GOOGL: -44.32%
   JPM: -43.63%
   VNQ: -42.40%


# Section 8: RISK METRIC — BETA vs S&P 500

In [23]:
# Beta measures a stock's SENSITIVITY to market movements.
# It compares the stock's returns to the S&P 500 (the market benchmark).
#
# Beta = 1.0 → Moves exactly with the market
# Beta > 1.0 → More volatile than the market (e.g., 1.5 = 50% more volatile)
# Beta < 1.0 → Less volatile than the market (defensive stocks)
# Beta < 0   → Moves opposite to the market (rare, e.g., gold in some periods)
#
# Beta is calculated as: Cov(stock, market) / Var(market)

print(f"\n Beta vs S&P 500:")

# Download S&P 500 data for the same period
sp500 = yf.download('^GSPC', start=START_DATE, end=END_DATE, auto_adjust=True, progress = False)['Close']
sp500_returns = np.log(sp500 / sp500.shift(1)).dropna()


# Note: Index ETFs (e.g. VOO, SPY, QQQ) will show beta close to 1.0 by definition
# since they track or closely follow the market benchmark.
# Beta is most meaningful for individual stocks.
betas = {}
for ticker in TICKERS:
    # Align dates between stock and S&P 500
    aligned = pd.concat([daily_returns[ticker], sp500_returns], axis=1).dropna()
    aligned.columns = ['stock', 'market']

    cov_matrix = np.cov(aligned['stock'], aligned['market'])
    beta = cov_matrix[0, 1] / cov_matrix[1, 1]  # Cov(stock,mkt) / Var(mkt)
    betas[ticker] = beta
    print(f"   {ticker}: β = {beta:.2f}")


 Beta vs S&P 500:
   SPY: β = 0.99
   TLT: β = -0.11
   GOOGL: β = 1.13
   JPM: β = 1.07
   VNQ: β = 0.89


# Section 9: CORRELATION MATRIX HEATMAP

In [24]:
# The correlation matrix shows how assets move RELATIVE TO EACH OTHER.
# Values range from -1 to +1:
#   +1.0 = Perfect positive correlation (always move together)
#    0.0 = No correlation (move independently)
#   -1.0 = Perfect negative correlation (always move opposite)
#
# For diversification, you WANT low or negative correlations.
# A portfolio of highly correlated assets doesn't reduce risk —
# they all fall together in a crash.

corr_matrix = daily_returns.corr()

fig_corr = go.Figure(data=go.Heatmap(
    z=corr_matrix.values,
    x=corr_matrix.columns.tolist(),
    y=corr_matrix.index.tolist(),
    colorscale='RdYlGn',
    zmin=-1, zmax=1,
    text=np.round(corr_matrix.values, 2),
    texttemplate='%{text}',
    textfont={"size": 14},
    hoverongaps=False
))

fig_corr.update_layout(
    title='Correlation Matrix — How Assets Move Together',
    title_font_size=18,
    width=600, height=500
)
fig_corr.show()

# Section 10: RISK SUMMARY DASHBOARD

In [25]:
# Compile all metrics into one clean summary table

summary = pd.DataFrame({
    'Annual Return': annual_returns.map('{:.1%}'.format),
    'Annual Volatility': annual_volatility.map('{:.1%}'.format),
    'Sharpe Ratio': sharpe_ratios.map('{:.2f}'.format),
    f'VaR ({VAR_CONFIDENCE:.0%})': pd.Series(var_values).map('{:.2%}'.format),
    f'CVaR ({VAR_CONFIDENCE:.0%})': pd.Series(cvar_values).map('{:.2%}'.format),
    'Max Drawdown': pd.Series(max_drawdowns).map('{:.2%}'.format),
    'Beta (vs S&P)': pd.Series(betas).map('{:.2f}'.format),
})

print("\n" + "="*70)
print("FULL RISK METRICS SUMMARY")
print(f"   Period: {START_DATE} to {END_DATE}")
print(f"   Risk-Free Rate: {RISK_FREE_RATE:.2%}")
print("="*70)
print(summary.to_string())
print("="*70)


FULL RISK METRICS SUMMARY
   Period: 2019-03-31 to 2026-03-31
   Risk-Free Rate: 3.61%
      Annual Return Annual Volatility Sharpe Ratio VaR (95%) CVaR (95%) Max Drawdown Beta (vs S&P)
GOOGL         21.9%             31.4%         0.58    -3.06%     -4.61%      -44.32%          1.13
JPM           17.1%             30.0%         0.45    -2.80%     -4.42%      -43.63%          1.07
SPY           12.9%             19.8%         0.47    -1.78%     -3.05%      -33.72%          0.99
TLT           -2.4%             16.5%        -0.36    -1.66%     -2.26%      -48.35%         -0.11
VNQ            3.9%             23.3%         0.01    -2.09%     -3.48%      -42.40%          0.89


# Section 11: MODERN PORTFOLIO THEORY — EFFICIENT FRONTIER

In [26]:
# Modern Portfolio Theory shows that you can create a portfolio that maximizes
# return for a given level of risk or minimizes risk for a given return.
#
# The EFFICIENT FRONTIER is the curve of these optimal portfolios.
# Any portfolio ON the frontier is "efficient" — you can't do better
# without taking on more risk.
#
# We find it by simulating thousands of random portfolios and
# plotting their risk (volatility) vs return.

# Note: runtime scales with NUM_PORTFOLIOS and number of tickers
# 10,000 portfolios with 5 assets runs in ~2-3 seconds
# Increase to 50,000 for a smoother frontier if needed

print(f"\n Simulating {NUM_PORTFOLIOS:,} random portfolios...")

mean_returns = daily_returns.mean() * 252       # annualized mean returns
cov_matrix_annual = daily_returns.cov() * 252   # annualized covariance matrix
n_assets = len(TICKERS)

# Store results
port_returns = []
port_volatilities = []
port_sharpes = []
port_weights = []

for _ in range(NUM_PORTFOLIOS):
    # Generate random weights that sum to 1 (100% invested)
    weights = np.random.random(n_assets)
    weights /= weights.sum()

    # Portfolio return = weighted average of individual returns
    p_return = np.dot(weights, mean_returns)

    # Portfolio volatility accounts for correlations between stocks
    # This is why diversification works — correlated assets don't
    # simply add their volatilities
    p_volatility = np.sqrt(weights.T @ cov_matrix_annual @ weights)

    p_sharpe = (p_return - RISK_FREE_RATE) / p_volatility

    port_returns.append(p_return)
    port_volatilities.append(p_volatility)
    port_sharpes.append(p_sharpe)
    port_weights.append(weights)

port_returns = np.array(port_returns)
port_volatilities = np.array(port_volatilities)
port_sharpes = np.array(port_sharpes)

print("Simulation complete!")


 Simulating 10,000 random portfolios...
Simulation complete!


# Section 12: FIND OPTIMAL PORTFOLIOS

In [27]:
# Two key portfolios on the efficient frontier:
#
# 1. MAX SHARPE RATIO PORTFOLIO ("Tangency Portfolio")
#    → Best risk-adjusted return. If you had to pick one portfolio,
#      most finance theory says this is it.
#
# 2. MINIMUM VARIANCE PORTFOLIO
#    → Lowest possible risk. Good for very risk-averse investors.

# Note: weights will vary slightly each run since portfolios are randomly simulated
# Run more portfolios (NUM_PORTFOLIOS) for more stable results

# Max Sharpe Portfolio
max_sharpe_idx = np.argmax(port_sharpes)
max_sharpe_return = port_returns[max_sharpe_idx]
max_sharpe_vol = port_volatilities[max_sharpe_idx]
max_sharpe_weights = port_weights[max_sharpe_idx]

# Min Variance Portfolio
min_var_idx = np.argmin(port_volatilities)
min_var_return = port_returns[min_var_idx]
min_var_vol = port_volatilities[min_var_idx]
min_var_weights = port_weights[min_var_idx]

print("\n OPTIMAL PORTFOLIOS:")
print(f"\n   Max Sharpe Ratio Portfolio (Sharpe = {port_sharpes[max_sharpe_idx]:.2f}):")
for i, ticker in enumerate(TICKERS):
    print(f"     {ticker}: {max_sharpe_weights[i]:.1%}")
print(f"   → Expected Return: {max_sharpe_return:.1%} | Volatility: {max_sharpe_vol:.1%}")

print(f"\n   Minimum Variance Portfolio:")
for i, ticker in enumerate(TICKERS):
    print(f"     {ticker}: {min_var_weights[i]:.1%}")
print(f"   → Expected Return: {min_var_return:.1%} | Volatility: {min_var_vol:.1%}")


 OPTIMAL PORTFOLIOS:

   Max Sharpe Ratio Portfolio (Sharpe = 0.61):
     SPY: 60.3%
     TLT: 22.9%
     GOOGL: 15.7%
     JPM: 0.9%
     VNQ: 0.3%
   → Expected Return: 19.1% | Volatility: 25.2%

   Minimum Variance Portfolio:
     SPY: 4.4%
     TLT: 8.1%
     GOOGL: 30.0%
     JPM: 57.1%
     VNQ: 0.4%
   → Expected Return: 4.9% | Volatility: 11.8%


# Section 13: PLOT THE EFFICIENT FRONTIER

In [28]:
fig_ef = go.Figure()

# Plot all simulated portfolios, colored by Sharpe Ratio
fig_ef.add_trace(go.Scatter(
    x=port_volatilities,
    y=port_returns,
    mode='markers',
    marker=dict(
        color=port_sharpes,
        colorscale='Viridis',
        size=4,
        opacity=0.6,
        colorbar=dict(title='Sharpe Ratio'),
        showscale=True
    ),
    text=[f"Sharpe: {s:.2f}" for s in port_sharpes],
    hovertemplate='Volatility: %{x:.1%}<br>Return: %{y:.1%}<br>%{text}',
    name='Simulated Portfolios'
))

# Highlight Max Sharpe portfolio
fig_ef.add_trace(go.Scatter(
    x=[max_sharpe_vol],
    y=[max_sharpe_return],
    mode='markers',
    marker=dict(color='red', size=16, symbol='star'),
    name=f'Max Sharpe (★)',
    hovertemplate=f'Max Sharpe Portfolio<br>Return: {max_sharpe_return:.1%}<br>Volatility: {max_sharpe_vol:.1%}'
))

# Highlight Min Variance portfolio
fig_ef.add_trace(go.Scatter(
    x=[min_var_vol],
    y=[min_var_return],
    mode='markers',
    marker=dict(color='blue', size=16, symbol='diamond'),
    name='Min Variance (◆)',
    hovertemplate=f'Min Variance Portfolio<br>Return: {min_var_return:.1%}<br>Volatility: {min_var_vol:.1%}'
))

# Plot individual stocks for reference
for i, ticker in enumerate(TICKERS):
    fig_ef.add_trace(go.Scatter(
        x=[annual_volatility[ticker]],
        y=[annual_returns[ticker]],
        mode='markers+text',
        marker=dict(color='white', size=12, symbol='circle',
                    line=dict(color='black', width=2)),
        text=[ticker],
        textposition='top center',
        name=ticker,
        showlegend=True
    ))

fig_ef.update_layout(
    title='Efficient Frontier — Risk vs Return by Asset and Portfolio',
    xaxis_title='Annual Volatility (Risk)',
    yaxis_title='Annual Expected Return',
    xaxis=dict(tickformat='.0%', range=[0, 0.40]),
    yaxis=dict(tickformat='.0%', range=[0, 0.30]),
    title_font_size=18,
    width=850,
    height=600,
    legend=dict(x=0.01, y=0.99),
    hovermode='closest'
)

# Capital Market Line — drawn from risk-free rate through Max Sharpe portfolio
cml_x = [0, max_sharpe_vol * 1.5]
cml_y = [RISK_FREE_RATE, RISK_FREE_RATE + (max_sharpe_return - RISK_FREE_RATE) * 1.5]
fig_ef.add_trace(go.Scatter(
    x=cml_x,
    y=cml_y,
    mode='lines',
    line=dict(color='red', dash='dash', width=2),
    name='Capital Market Line'
))
fig_ef.show()

# Section 14: PORTFOLIO WEIGHTS PIE CHARTS

In [29]:
fig_pie = make_subplots(
    rows=1, cols=2,
    specs=[[{'type': 'pie'}, {'type': 'pie'}]],
    subplot_titles=['Max Sharpe Portfolio', 'Min Variance Portfolio']
)

fig_pie.add_trace(go.Pie(
    labels=TICKERS,
    values=np.round(max_sharpe_weights, 3),
    hole=0.4,
    textinfo='label+percent',
    hovertemplate='%{label}<br>Weight: %{percent}<extra></extra>'
), row=1, col=1)

fig_pie.add_trace(go.Pie(
    labels=TICKERS,
    values=np.round(min_var_weights, 3),
    hole=0.4,
    textinfo='label+percent',
    hovertemplate='%{label}<br>Weight: %{percent}<extra></extra>'
), row=1, col=2)

fig_pie.update_layout(
    title='Optimal Portfolio Allocations - Max Sharpe vs Min Variance',
    title_font_size=18,
    width=850,
    height=450
)
fig_pie.show()

# Section 15: CUMULATIVE RETURNS CHART

In [30]:
# This shows how $1 invested in each stock at the start
# would have grown over the entire period — useful for
# visualizing historical performance alongside risk metrics

cumulative_returns = prices / prices.iloc[0]

fig_cum = go.Figure()
for ticker in TICKERS:
    fig_cum.add_trace(go.Scatter(
        x=cumulative_returns.index,
        y=cumulative_returns[ticker],
        mode='lines',
        name=ticker,
        hovertemplate=f'{ticker}<br>Date: %{{x}}<br>Growth of $1: $%{{y:.2f}}'
    ))

fig_cum.update_layout(
    title='Cumulative Returns — Growth of $1 Invested',
    xaxis_title='Date',
    yaxis_title='Portfolio Value (starting at $1)',
    title_font_size=18,
    width=850,
    height=500,
    hovermode='x unified'
)

fig_cum.add_vrect(
    x0='2020-02-19', x1='2020-03-23',
    fillcolor='red', opacity=0.2,
    annotation_text='COVID Crash', annotation_position='top left'
)
fig_cum.add_vrect(
    x0='2022-01-03', x1='2022-10-12',
    fillcolor='orange', opacity=0.1,
    annotation_text='2022 Bear Market', annotation_position='top left'
)
fig_cum.add_vrect(
    x0='2025-01-01', x1='2025-04-01',
    fillcolor='purple', opacity=0.1,
    annotation_text='2025 Tariff Selloff', annotation_position='top left'
)

fig_cum.show()

print("\n Analysis complete! Scroll up to view all charts and metrics.")
print(" Tip: Edit TICKERS in Section 2 to analyze your own portfolio. Supports stocks, ETFs, and bonds")


 Analysis complete! Scroll up to view all charts and metrics.
 Tip: Edit TICKERS in Section 2 to analyze your own portfolio. Supports stocks, ETFs, and bonds


#